# Halos Revisited

The previous implementation is as fast as a single GPU - independent of the number of GPUs actually used.
Visualizing the time line with the Nsight Systems GUI reveals the culprit.
Our device-to-device transfers are blocking, even though we used cudaMemcpyAsync.

The reason is the stream semantic.
Operations enqueued in the default stream, such as our copies, block all other operations while they are running and force operation-level synchronization.
This extends to the default streams of **both** devices involved.
Or, in other words, our transfers block sending and receiving devices, even though each one has its own default stream.

The fix is simple - we can just use a separate stream.

Note: `cudaStreamCreate` ties the new stream to the currently active device.
Make sure to use `cudaSetDevice` **before** creating each stream.

## Exercise

This time there is only one difficulty level.

Create and empty file [stencil-2d.cu](../src/07-halos-2/stencil-2d.cu) and copy one of the previous code versions of into it (your work or a solution).
Then apply the fix discussed above.

In [ ]:
!touch ../src/07-halos-2/stencil-2d.cu

### Possible Solution

[stencil-2d-solution.cu](../src/07-halos-2/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/07-halos-2/stencil-2d.cu) with the cell below.

In [ ]:
!cp ../src/07-halos-2/stencil-2d-solution.cu ../src/07-halos-2/stencil-2d.cu

### Compilation, Execution and Profiling

The new code version is available in [07-halos-2/stencil-2d.cu](../src/07-halos-2/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [ ]:
!nvcc -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/07-halos-2 ../src/07-halos-2/stencil-2d.cu

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.

In [ ]:
!../build/07-halos-2 256 64 2 2000 100

The next cell produces no output and runs a larger grid.
Use it for performance evaluation.

In [ ]:
!../build/07-halos-2 $((32 * 1024)) 256 2 16 0

Instead of running on the local **single A40** GPU, we can also submit a batch job running on **multiple A100** GPUs.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between one and eight GPUs.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/07-halos-2.out --error=../output/07-halos-2.err \
    --wrap="../build/07-halos-2 $((32 * 1024)) 256 2 8 0"

cat ../output/07-halos-2.out

The next cell performs profiling with Nsight Systems by submitting a batch job.
Feel free to tune the number of GPUs in the `--gres=gpu:a100:NGPU` to anything between one and eight GPUs.

The profile is then available at [profiles/07-halos-2.nsys-rep](../profiles/07-halos-2.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/07-halos-2-nsys.out --error=../output/07-halos-2-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/07-halos-2 \
        ../build/07-halos-2 $((32 * 1024)) 256 2 8 0"

cat ../output/07-halos-2-nsys.out

## Next Step

The next step is optimizing device-to-device transfers in [08](./08-p2p.ipynb).